# T16 — Learn from Data: Profile, Infer, Generate

Profile existing data to infer a Spindle schema, then generate statistically similar synthetic data.

**What you'll learn:**
- Profile DataFrames to detect types, distributions, PKs, and FKs
- Build a generation schema from the profile
- Generate synthetic data that mirrors the original distributions

## 1. Create Sample Real Data

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

customers = pd.DataFrame({
    "customer_id": range(1, 501),
    "email": [f"user{i}@example.com" for i in range(1, 501)],
    "tier": rng.choice(["basic", "silver", "gold", "platinum"], 500, p=[0.5, 0.25, 0.15, 0.1]),
    "age": rng.normal(38, 12, 500).astype(int).clip(18, 90),
    "annual_spend": rng.lognormal(7, 1.2, 500).round(2),
})

orders = pd.DataFrame({
    "order_id": range(1, 2001),
    "customer_id": rng.choice(range(1, 501), 2000),
    "amount": rng.lognormal(4, 0.8, 2000).round(2),
    "status": rng.choice(["completed", "shipped", "cancelled"], 2000, p=[0.8, 0.15, 0.05]),
})

print(f"Customers: {len(customers)} rows, Orders: {len(orders)} rows")
customers.head()

Customers: 500 rows, Orders: 2000 rows


,customer_id,email,tier,age,annual_spend
0,1,user1@example.com,gold,55,174.71
1,2,user2@example.com,basic,39,1444.30
2,3,user3@example.com,gold,44,2651.02
3,4,user4@example.com,silver,37,1718.60
4,5,user5@example.com,basic,35,2341.10


## 2. Profile the Dataset

In [2]:
from sqllocks_spindle.inference import DataProfiler

profiler = DataProfiler()
profile = profiler.profile_dataset({"customer": customers, "order": orders})

for table_name, table_profile in profile.tables.items():
    print(f"\n=== {table_name} ===")
    for col in table_profile.columns.values():
        pk = " [PK]" if col.is_primary_key else ""
        fk = f" [FK -> {col.fk_ref_table}]" if col.is_foreign_key else ""
        enum = f" (enum: {len(col.enum_values)} values)" if col.is_enum else ""
        dist = f" (dist: {col.distribution})" if col.distribution else ""
        print(f"  {col.name:20s} {col.dtype:10s}{pk}{fk}{enum}{dist}")


=== customer ===
  customer_id          integer    [PK] (dist: uniform)
  email                string    
  tier                 string     (enum: 4 values)
  age                  integer    (dist: lognormal)
  annual_spend         float      (dist: lognormal)

=== order ===
  order_id             integer    [PK] (dist: uniform)
  customer_id          integer    [FK -> customer] (dist: uniform)
  amount               float      (dist: lognormal)
  status               string     (enum: 3 values)


## 3. Build Schema from Profile

In [3]:
from sqllocks_spindle.inference import SchemaBuilder

builder = SchemaBuilder()
schema = builder.build(profile, domain_name="my_retail")

print(f"Tables: {list(schema.tables.keys())}")
print(f"Relationships: {len(schema.relationships)}")
for t_name, t_def in schema.tables.items():
    print(f"\n{t_name}:")
    for c_name, c_def in t_def.columns.items():
        print(f"  {c_name:20s} -> {c_def.generator.get('strategy', '?')}")

Tables: ['customer', 'order']
Relationships: 1

customer:
  customer_id          -> sequence
  email                -> faker
  tier                 -> weighted_enum
  age                  -> distribution
  annual_spend         -> distribution

order:
  order_id             -> sequence
  customer_id          -> foreign_key
  amount               -> distribution
  status               -> weighted_enum


## 4. Generate Synthetic Data

In [4]:
from sqllocks_spindle import Spindle

result = Spindle().generate(schema=schema, scale="small", seed=42)
print(result.summary())

errors = result.verify_integrity()
print(f"\nFK integrity errors: {len(errors)}")

Spindle Generation Result
Schema: my_retail_inferred
Domain: my_retail
Mode:   3nf
Seed:   42
Time:   0.2s

Table                             Rows  Columns
---------------------------------------------
customer                           500        5
order                            2,000        4
---------------------------------------------
TOTAL                            2,500

FK integrity errors: 0


## 5. Compare Distributions

In [5]:
synth_customers = result["customer"]

print("=== Tier Distribution ===")
print("Real:")
print(customers["tier"].value_counts(normalize=True).sort_index())
if "tier" in synth_customers.columns:
    print("\nSynthetic:")
    print(synth_customers["tier"].value_counts(normalize=True).sort_index())

=== Tier Distribution ===
Real:
tier
basic       0.498
gold        0.158
platinum    0.090
silver      0.254
Name: proportion, dtype: float64

Synthetic:
tier
basic       0.498
gold        0.162
platinum    0.084
silver      0.256
Name: proportion, dtype: float64


## 6. CLI Equivalent

```bash
spindle learn ./real_data/ --format csv --output inferred.spindle.json --domain my_retail
spindle generate --schema inferred.spindle.json --scale small --output ./synthetic/
```